In [1]:
#import packages

import os
import re
import pandas as pd
import sklearn as sk
import numpy as np

In [2]:
#helper functions
def compute_scores(df_pred, df_gold, column_pred, column_gold, column_match):
    df_pred_reduced = df_pred[[column_match, column_pred]]
    df_gold_reduced = df_gold[[column_match, column_gold]]

    #join these two dataframes
    df_temp = (
        df_pred_reduced
            .merge(df_gold_reduced, on=column_match, how='inner', suffixes = ('_pred', '_gold'))   # keep only matching IDs
            .dropna(subset=[column_pred + '_pred', column_gold + '_pred'])          # drop rows where values are NaN
            .reset_index(drop=True)                                 # tidy up the index
    )
    
    #ensure that the columns have data in the same type 
    df_temp[column_pred + '_pred'] = df_temp[column_pred + '_pred'].astype(int)
    df_temp[column_gold + '_gold'] = df_temp[column_gold + '_gold'].astype(int)

    #compute scores
    acc = round(100*sk.metrics.accuracy_score(df_temp[column_gold + '_gold'], df_temp[column_pred + '_pred']), 3)
    k = round(sk.metrics.cohen_kappa_score(df_temp[column_gold + '_gold'], df_temp[column_pred + '_pred']), 3)
    f1 = round(sk.metrics.f1_score(df_temp[column_gold + '_gold'], df_temp[column_pred + '_pred']), 3)
    precision = round(sk.metrics.precision_score(df_temp[column_gold + '_gold'], df_temp[column_pred + '_pred']), 3)
    recall = round(sk.metrics.recall_score(df_temp[column_gold + '_gold'], df_temp[column_pred + '_pred']), 3)

    print("Accuracy for " + column_gold +  " is " + str(acc) + "%, and Cohen's kappa is " + str(k))

    return([acc, k, f1, precision, recall])

In [ ]:
#read all model names
files = os.listdir("Predicted/Test/")
models = list()
for file in files:
    if not re.search("scores", file) and not re.search(".DS_Store", file):
        models.append(file)


In [ ]:
#load data
df_gold = pd.read_csv('labelled_frames_test.csv')
frame_names = df_gold.columns[7:14].to_list()

for m in models:
    df = pd.read_csv('Predicted/Test/' + m)
    #scores
    scores_df = pd.DataFrame(index = frame_names,
                         columns= ['Frequency','Accuracy', 'Kappa', 'F1', 'Precision', 'Recall'])

    for f in frame_names:
        scores_df.loc[f, ['Accuracy', 'Kappa', 'F1', 'Precision', 'Recall']] = compute_scores(df_gold = df_gold, df_pred = df, column_match= "stories_id", column_gold= f, column_pred= f)
        scores_df.loc[f,'Frequency'] = df_gold[f].to_list().count(1)
    
    print(m)
    print(scores_df)
    scores_df.loc['mean'] = [round(elem, 3) for elem in scores_df.mean()]
    
    scores_df.to_csv('Scores/'+ m)

Accuracy for sexual stigma and transmission routes is 69.388%, and Cohen's kappa is 0.422
Accuracy for racial disparities and stigmatising name is 95.918%, and Cohen's kappa is 0.81
Accuracy for global relations is 71.429%, and Cohen's kappa is 0.244
Accuracy for public health failure is 81.633%, and Cohen's kappa is 0.541
Accuracy for epidemic preparedness and surveillance is 58.163%, and Cohen's kappa is 0.248
Accuracy for human-interest stories is 76.531%, and Cohen's kappa is 0.345
Accuracy for broader health issues is 72.449%, and Cohen's kappa is 0.384
llama3.1:8b-instruct-q8_0_human_judge.csv
                                         Frequency Accuracy  Kappa     F1  \
sexual stigma and transmission routes           38   69.388  0.422  0.694   
racial disparities and stigmatising name        12   95.918   0.81  0.833   
global relations                                10   71.429  0.244  0.364   
public health failure                           19   81.633  0.541  0.654   
epidemic

In [16]:
#save all model results in one file per metric
metrics = ['Accuracy', 'Kappa', 'F1', 'Precision', 'Recall']

for met in metrics:
    df_met = pd.DataFrame(index= models,
                          columns= frame_names)

    for m in models:
        df = pd.read_csv('Scores/'+m)
        df_met.loc[m,:] = df.loc[range(len(frame_names)),met].to_numpy()

    df_met.to_csv("Scores/All_"+met+".csv")
